# Notebook 2: Extracción y Preprocesamiento - Dr. Stephen Stahl (Prescriber's Guide)

Este notebook procesa las fichas técnicas y monografías de la **"Guía del Prescriptor" de Stephen Stahl**.

### Objetivos:
1. Extraer los campos estandarizados de las monografías de Stahl (Clase, Mecanismo receptor, Perfil de citocromos CYP450, Efectos frecuentes vs graves, Perlas clínicas).
2. Modelar con precisión el **perfil de enzimas hepáticas (CYP450)**: fundamental para calcular interacciones farmacocinéticas en polifarmacia.
3. Clasificar los eventos adversos por nivel de severidad (comunes vs potencialmente letales).
4. Exportar el catálogo limpio a CSV para la base de datos relacional SQLite.


In [2]:
import os
import sys
import re
from pathlib import Path
import pandas as pd
import numpy as np

# Detectar la ruta raíz del proyecto de forma dinámica y robusta
def _encontrar_raiz():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if (p / 'src').is_dir() and (p / 'data').is_dir():
            return p
        if (p / 'Sofia' / 'src').is_dir() and (p / 'Sofia' / 'data').is_dir():
            return p / 'Sofia'
    raise RuntimeError("No se pudo determinar la raíz del proyecto Sofia.")

project_root = _encontrar_raiz()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_RAW = project_root / 'data' / 'raw'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

from src.extractors.stahl_extractor import StahlExtractor

print('Entorno de Stahl configurado. Directorio de datos: data/processed')

Entorno de Stahl configurado. Directorio de datos: data/processed


## 1. Carga de Datos y Estructura Monográfica de Stahl

La Guía del Prescriptor de Stahl tiene un formato sistemático por fármaco. Definimos un lector con un corpus representativo de psicofármacos.


In [4]:
# Carga estructurada de monografías de Stahl mediante el extractor modular
stahl_extractor = StahlExtractor(raw_dir=DATA_RAW)
df_stahl, df_eventos_stahl = stahl_extractor.extract_monographs()

print(f'Total de fármacos psiquiátricos procesados (Stahl): {len(df_stahl)}')
print(f'Total de eventos adversos clasificados: {len(df_eventos_stahl)}')
df_stahl[['farmaco', 'clase', 'atc_codigo', 'inhibe_cyp2d6', 'potencia_inhibicion_2d6', 'inhibe_cyp1a2', 'inhibe_cyp3a4', 'sustrato_cyp1a2', 'sustrato_cyp2b6', 'sustrato_cyp2c19', 'sustrato_cyp3a4']].head(10)

Total de fármacos psiquiátricos procesados (Stahl): 22
Total de eventos adversos clasificados: 188


## 2. Segmentación de Monografías y Parser

Parseamos cada ficha monográfica de Stahl delimitada entre `[DRUG_START]` y `[DRUG_END]`.


In [6]:
# Estructura y columnas clínicas de las monografías de Stahl
print('Columnas clínicas y farmacocinéticas en el dataset de Stahl:')
for col in df_stahl.columns:
    print(f' - {col}: {str(df_stahl[col].iloc[0])[:70]}...')

Columnas clínicas y farmacocinéticas en el dataset de Stahl:
 - farmaco: Clozapina...
 - clase: Antipsicótico Atípico (Segunda Generación)...
 - perfil_receptores: Antagonista 5-HT2A > D2; alta afinidad por receptores D4, 5-HT2C, alfa...
 - metabolismo_cyp: Sustrato principal de CYP1A2. Sustrato secundario de CYP2D6 y CYP3A4. ...
 - efectos_comunes: Sedación diurna intensa, Sialorrea nocturna profusa, Aumento marcado d...
 - efectos_graves: Agranulocitosis (neutropenia grave potencialmente mortal <1000/µL), Mi...
 - perlas_clinicas: Indicado en esquizofrenia resistente y riesgo suicida recurrente. Requ...
 - inhibe_cyp1a2: 0...
 - inhibe_cyp2c9: 0...
 - inhibe_cyp2c19: 0...
 - inhibe_cyp2d6: 0...
 - potencia_inhibicion_2d6: Nula...
 - inhibe_cyp3a4: 0...
 - sustrato_cyp1a2: 1...
 - sustrato_cyp2b6: 0...
 - sustrato_cyp2c9: 0...
 - sustrato_cyp2c19: 0...
 - sustrato_cyp2d6: 1...
 - sustrato_cyp3a4: 1...
 - inductor_cyp1a2: 0...
 - inductor_cyp2c9: 0...
 - inductor_cyp2c19: 0...
 - induc

## 3. Extracción de Features Farmacocinéticas: Citocromos CYP450

En Stahl, el comportamiento frente al citocromo P450 es crucial. Creamos flags booleanos/categóricos para identificar si el fármaco inhibe enzimas como **CYP2D6, CYP1A2 o CYP3A4**.


In [8]:
# Aserciones de validación farmacocinética clínica
assert len(df_stahl) == 22, f'Error: Se esperaban 22 fármacos en Stahl, encontrados {len(df_stahl)}'
assert df_stahl['atc_codigo'].notnull().all(), 'Error: Hay fármacos sin código ATC'
assert df_stahl.loc[df_stahl['farmaco'] == 'Paroxetina', 'inhibe_cyp2d6'].values[0] == 1, 'Error: Paroxetina debe inhibir CYP2D6'
assert df_stahl.loc[df_stahl['farmaco'] == 'Bupropión', 'inhibe_cyp2d6'].values[0] == 1, 'Error: Bupropión debe inhibir CYP2D6'
assert df_stahl.loc[df_stahl['farmaco'] == 'Bupropión', 'sustrato_cyp2b6'].values[0] == 1, 'Error: Bupropión debe ser sustrato de CYP2B6'
assert df_stahl.loc[df_stahl['farmaco'] == 'Clozapina', 'sustrato_cyp1a2'].values[0] == 1, 'Error: Clozapina debe ser sustrato de CYP1A2'
assert df_stahl.loc[df_stahl['farmaco'] == 'Carbamazepina', 'inductor_cyp3a4'].values[0] == 1, 'Error: Carbamazepina debe inducir CYP3A4'
assert df_stahl.loc[df_stahl['farmaco'] == 'Escitalopram', 'sustrato_cyp2c19'].values[0] == 1, 'Error: Escitalopram debe ser sustrato de CYP2C19'

# Comprobar fármacos no dependientes de metabolismo oxidativo hepático
cols_cyp = ['inhibe_cyp2d6', 'inhibe_cyp3a4', 'sustrato_cyp1a2', 'sustrato_cyp2b6', 'sustrato_cyp2d6', 'sustrato_cyp3a4']
assert df_stahl.loc[df_stahl['farmaco'] == 'Lamotrigina', cols_cyp].sum().sum() == 0, 'Error: Lamotrigina no debe tener marcas CYP oxidativas'
assert df_stahl.loc[df_stahl['farmaco'] == 'Carbonato de Litio', cols_cyp].sum().sum() == 0, 'Error: Litio no debe tener marcas CYP oxidativas'

print('✓ Todas las aserciones de farmacocinética y perfiles CYP450 validadas con éxito.')

✓ Todas las aserciones de farmacocinética y perfiles CYP450 validadas con éxito.


## 4. Desagregación de Efectos Adversos por Severidad

Stahl distingue claramente entre efectos secundarios tolerables y eventos adversos con riesgo de muerte. Creamos la tabla normalizada con el atributo `severidad`.


In [10]:
# Control de calidad de eventos adversos desagregados por severidad
print('Distribución de eventos adversos de Stahl por severidad:')
print(df_eventos_stahl['severidad'].value_counts())

# Validar que no existan textos cortados con paréntesis huérfanos ni puntos sueltos
par_abiertos = df_eventos_stahl['efecto_adverso'].str.count(r'\(')
par_cerrados = df_eventos_stahl['efecto_adverso'].str.count(r'\)')
assert (par_abiertos == par_cerrados).all(), 'Error: Paréntesis desbalanceados en eventos adversos'

assert not df_eventos_stahl['efecto_adverso'].str.endswith('.').any(), 'Error: Hay eventos adversos con punto final suelto'
assert df_eventos_stahl['efecto_adverso'].apply(lambda s: s[0].isupper()).all(), 'Error: Hay eventos adversos que no inician en mayúscula'

print(f'Total eventos clínicos verificados: {len(df_eventos_stahl)}')
df_eventos_stahl.head(10)

Distribución de eventos adversos de Stahl por severidad:
severidad
Común / Moderado        116
Grave / Riesgo Vital     72
Name: count, dtype: int64
Total eventos clínicos verificados: 188


## 5. Exportación de Resultados

Guardamos los datasets limpios para integración con SQLite.


In [12]:
out_farmacos = DATA_PROCESSED / 'stahl_farmacos_clean.csv'
out_eventos = DATA_PROCESSED / 'stahl_eventos_adversos_clean.csv'

# Guardar con codificación UTF-8
df_stahl_export = df_stahl.copy()
if 'efectos_comunes' in df_stahl_export.columns and isinstance(df_stahl_export['efectos_comunes'].iloc[0], list):
    df_stahl_export['efectos_comunes'] = df_stahl_export['efectos_comunes'].apply(lambda l: ', '.join(l))
if 'efectos_graves' in df_stahl_export.columns and isinstance(df_stahl_export['efectos_graves'].iloc[0], list):
    df_stahl_export['efectos_graves'] = df_stahl_export['efectos_graves'].apply(lambda l: ', '.join(l))

df_stahl_export.to_csv(out_farmacos, index=False, encoding='utf-8')
df_eventos_stahl.to_csv(out_eventos, index=False, encoding='utf-8')

rel_farmacos = out_farmacos.relative_to(project_root)
rel_eventos = out_eventos.relative_to(project_root)
print(f'Fármacos de Stahl guardados en: {rel_farmacos} ({len(df_stahl)} fármacos, 16 columnas CYP)')
print(f'Eventos adversos de Stahl guardados en: {rel_eventos} ({len(df_eventos_stahl)} eventos)')

Fármacos de Stahl guardados en: data/processed/stahl_farmacos_clean.csv (22 fármacos, 16 columnas CYP)
Eventos adversos de Stahl guardados en: data/processed/stahl_eventos_adversos_clean.csv (188 eventos)
